In [1]:
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, select, update
from sqlalchemy.orm import sessionmaker
from models import Comment
from filter import *

In [2]:
load_dotenv()

db_host = os.getenv('db_host')
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

In [3]:
DATABASE_URL = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

In [4]:
with open('security_keywords.txt', 'r') as f:
    security_keywords = [line.strip() for line in f.readlines() if line.strip()]

print(security_keywords)

['TOCTOU', 'breach', 'code injection', 'cross site', 'ddos', 'dead lock', 'dead-lock', 'deadlock', 'deadlocks', 'denial of service', 'exploit', 'forged', 'gain access', 'infinite recursion', 'infinite-recursion', 'leak', 'malicious', 'malicious code', 'overflow', 'overrun', 'race', 'races', 'racy', 'redos', 'sql injection', 'stack overflow', 'unauthenticated', 'underflow', 'vulnerability', 'vulnerable', 'xsrf', 'xss', 'xxe', 'zip slip', 'zipslip']


In [5]:
# comments = []

# with Session() as session:
#     # select only the columns we need to reduce payload and enable streaming
#     qry = select(
#         Comment.id,
#         Comment.body,
#         Comment.author,
#         Comment.diff_hunk,
#         Comment.pr_id,
#         Comment.owner,
#         Comment.name,
#         Comment.original_commit_id,
#     ).where(Comment.possible_response == False).where(Comment.in_reply_to_id.is_(None))

#     # stream results so we don't load all rows into memory at once
#     result = session.execute(qry.execution_options(stream_results=True))

#     counter = 0

#     for row in result:
#         print(f"Processing comment {counter}", end='\r')
#         counter += 1
#         body = row.body
#         con, k = contains_keyword(body, security_keywords)
#         if con:
#             comments.append({
#                 'id': row.id,
#                 'body': body,
#                 'author': row.author,
#                 'diff': row.diff_hunk,
#                 'keywords': k,
#                 'pr_id': row.pr_id,
#                 'owner': row.owner,
#                 'name': row.name,
#                 'url': f'https://github.com/{row.owner}/{row.name}/pull/{row.pr_id}/files/{row.original_commit_id}'
#             })

# print(f"Total comments with security keywords: {len(comments)}")
    

In [9]:
# assumes: contains_keyword(text, security_keywords) -> (bool, list_of_keywords)
# and SQLAlchemy models: Comment, Session

BATCH_SIZE = 5_000  # tune to your RAM/IO profile

def fetch_comments_in_batches(session, batch_size=BATCH_SIZE):
    """
    Generator that yields rows of Comment in batches using keyset pagination on Comment.id.
    Respects your filters: possible_response == False and in_reply_to_id is NULL.
    """
    last_id = None
    base_filters = [
        Comment.possible_response.is_(False),
        Comment.in_reply_to_id.is_(None),
    ]

    cols = (
        Comment.id,
        Comment.body,
        Comment.author,
        Comment.diff_hunk,
        Comment.pr_id,
        Comment.owner,
        Comment.name,
        Comment.original_commit_id,
        Comment.path,
    )

    while True:
        filters = list(base_filters)
        if last_id is not None:
            filters.append(Comment.id > last_id)

        qry = (
            select(*cols)
            .where(*filters)
            .order_by(Comment.id)
            .limit(batch_size)
        )

        batch = session.execute(qry).all()
        if not batch:
            break

        # advance cursor for next page
        last_id = batch[-1][0]  # first column is Comment.id

        # yield the batch to caller
        for row in batch:
            yield row

comments = []
counter = 0
matched = 0

with Session() as session:
    for row in fetch_comments_in_batches(session):
        # unpack by position to keep it fast; row is a tuple in the same order as cols
        (
            comment_id,
            body,
            author,
            diff_hunk,
            pr_id,
            owner,
            name,
            original_commit_id,
            path,
        ) = row

        counter += 1
        if counter % 5000 == 0:
            print(f"Scanned {counter} comments...", end="\r")

        con, k = contains_keyword(body, security_keywords)
        if con:
            matched += 1
            comments.append({
                'id': comment_id,
                'body': body,
                'author': author,
                'diff': diff_hunk,
                'keywords': k,
                'pr_id': pr_id,
                'owner': owner,
                'name': name,
                'url': f'https://github.com/{owner}/{name}/pull/{pr_id}/files/{original_commit_id}',
                'path': path,
            })

print(f"Total scanned: {counter} | Total comments with security keywords: {matched}")


Total scanned: 113552 | Total comments with security keywords: 400


In [10]:
comments[0]

{'id': 4183230,
 'body': "This is a race-condition and wouldn't work (I believe) ... but it's actually made redundant and will never get applied since `completed` will only ever equal `observers.size()` once since `completed` atomically increases each time it's called.\n",
 'author': 'benjchristensen',
 'diff': '@@ -174,18 +161,15 @@ public Aggregator(FuncN<R> combineLatestFunction) {\n          * \n          * @param w The observer that has completed.\n          */\n-        <T> void complete(CombineObserver<R, T> w) {\n-            synchronized(lockObject) {\n-                // store that this CombineLatestObserver is completed\n-                completed.add(w);\n-                // if all CombineObservers are completed, we mark the whole thing as completed\n-                if (completed.size() == observers.size()) {\n-                    if (running.get()) {\n-                        // mark ourselves as done\n-                        observer.onCompleted();\n-                   

In [11]:
print(len(comments))

400


In [13]:
counter = 0
for c in comments:
    if c['path'].endswith('.java'):
        counter += 1
    else:
        print(c['id'])
print(counter)

39599150
39607857
39645855
46444714
62824551
110096800
110097680
190510374
302012617
302013406
315095155
321478644
339961236
339965028
366852637
374951022
377222541
377222844
427986740
433423040
433485552
433485779
433486466
433863377
433865085
433870499
433871821
433876633
433876993
433877398
433877852
458218713
459098256
465671338
470623389
470681609
477534587
533439622
533442499
555077427
573996491
599209785
599210812
699303649
709791131
722978342
736530346
745601728
761513956
766814545
915584601
920495941
966834116
972001591
982462519
1552304047
1672239396
343


In [8]:
for comment in comments:
    if comment['pr_id'] == 46103:  # Replace with an actual comment ID to test
        print(comment)

{'id': 319010097, 'body': 'I am wondering if for this case we should first make `docVectorMagnitude` double to allow enough precision and avoid overflow as we do for other calculations? After that we can `sqrt` it and make float', 'author': 'mayya-sharipova', 'diff': '@@ -108,28 +127,35 @@ public CosineSimilarity(ScoreScript scoreScript, List<Number> queryVector) {\n         }\n \n         public double cosineSimilarity(VectorScriptDocValues.DenseVectorScriptDocValues dvs) {\n-            BytesRef value = dvs.getEncodedValue();\n-            float[] docVector = VectorEncoderDecoder.decodeDenseVector(scoreScript._getIndexVersion(), value);\n-            if (queryVector.size() != docVector.length) {\n+            BytesRef vector = dvs.getEncodedValue();\n+            int vectorLength = VectorEncoderDecoder.denseVectorLength(scoreScript._getIndexVersion(), vector);\n+            if (queryVector.size() != vectorLength) {\n                 throw new IllegalArgumentException("Can\'t calculat

In [13]:
# Save to excel
import pandas as pd
df = pd.DataFrame(comments)
df.to_excel('real_dataset_with_security_keywords.xlsx', index=False)